**1. Calculate Spearman rank correlation coefficient**

In [1]:
import pandas as pd

# ---------------------------------------------------------
# Example daily returns of 5 assets
# ---------------------------------------------------------

returns = pd.DataFrame({
    "Stock_A": [0.010, 0.015, -0.005, 0.008, 0.012,
                -0.010, 0.006, 0.009, -0.004, 0.011],

    "Stock_B": [0.009, 0.014, -0.004, 0.007, 0.011,
                -0.009, 0.005, 0.008, -0.003, 0.010],

    "Stock_C": [-0.005, 0.008, 0.012, -0.010, 0.006,
                0.004, -0.007, 0.011, -0.002, 0.009],

    "Stock_D": [-0.008, -0.012, 0.005, -0.006, -0.010,
                0.009, -0.004, -0.007, 0.003, -0.009],

    "Stock_E": [0.002, -0.004, 0.006, 0.001, -0.003,
                0.005, -0.002, 0.004, 0.000, -0.001]
})


# ---------------------------------------------------------
# Spearman Rank Correlation Matrix
# ---------------------------------------------------------

spearman_matrix = returns.corr(method="spearman")

print("Spearman Rank Correlation Matrix:")
print(spearman_matrix)

Spearman Rank Correlation Matrix:
          Stock_A   Stock_B   Stock_C   Stock_D   Stock_E
Stock_A  1.000000  1.000000  0.127273 -1.000000 -0.721212
Stock_B  1.000000  1.000000  0.127273 -1.000000 -0.721212
Stock_C  0.127273  0.127273  1.000000 -0.127273  0.224242
Stock_D -1.000000 -1.000000 -0.127273  1.000000  0.721212
Stock_E -0.721212 -0.721212  0.224242  0.721212  1.000000


**2. Make a diversification decision**

In [2]:
def diversification_decision(rho):

    if rho > 0.70:
        return (
            "LIMITED DIVERSIFICATION",
            "Assets tend to move in the same rank direction."
        )

    elif rho > 0.30:
        return (
            "MODERATE DIVERSIFICATION",
            "Assets have a moderate positive relationship."
        )

    elif rho >= -0.30:
        return (
            "GOOD DIVERSIFICATION",
            "Assets have a weak monotonic relationship."
        )

    else:
        return (
            "STRONG DIVERSIFICATION",
            "Assets tend to move in opposite rank directions."
        )

**3. Analyze every asset pair**

In [3]:
assets = spearman_matrix.columns

results = []

for i in range(len(assets)):

    for j in range(i + 1, len(assets)):

        asset_1 = assets[i]
        asset_2 = assets[j]

        rho = spearman_matrix.loc[asset_1, asset_2]

        decision, explanation = diversification_decision(rho)

        results.append({
            "Asset_1": asset_1,
            "Asset_2": asset_2,
            "Spearman_Rho": round(rho, 3),
            "Decision": decision,
            "Explanation": explanation
        })


results_df = pd.DataFrame(results)

print(results_df)

   Asset_1  Asset_2  Spearman_Rho                 Decision  \
0  Stock_A  Stock_B         1.000  LIMITED DIVERSIFICATION   
1  Stock_A  Stock_C         0.127     GOOD DIVERSIFICATION   
2  Stock_A  Stock_D        -1.000   STRONG DIVERSIFICATION   
3  Stock_A  Stock_E        -0.721   STRONG DIVERSIFICATION   
4  Stock_B  Stock_C         0.127     GOOD DIVERSIFICATION   
5  Stock_B  Stock_D        -1.000   STRONG DIVERSIFICATION   
6  Stock_B  Stock_E        -0.721   STRONG DIVERSIFICATION   
7  Stock_C  Stock_D        -0.127     GOOD DIVERSIFICATION   
8  Stock_C  Stock_E         0.224     GOOD DIVERSIFICATION   
9  Stock_D  Stock_E         0.721  LIMITED DIVERSIFICATION   

                                        Explanation  
0   Assets tend to move in the same rank direction.  
1        Assets have a weak monotonic relationship.  
2  Assets tend to move in opposite rank directions.  
3  Assets tend to move in opposite rank directions.  
4        Assets have a weak monotonic relations

4. Automatically select diversified assets

You can also create a function that selects assets while avoiding highly correlated assets.

In [4]:
def select_diversified_assets(
    returns,
    correlation_threshold=0.70
):

    # Calculate Spearman correlation
    corr_matrix = returns.corr(method="spearman")

    selected_assets = []

    for asset in corr_matrix.columns:

        # Select first asset
        if len(selected_assets) == 0:
            selected_assets.append(asset)
            continue

        # Correlation with already selected assets
        correlations = corr_matrix.loc[
            asset,
            selected_assets
        ]

        # Highest absolute Spearman correlation
        max_correlation = correlations.abs().max()

        # Decision
        if max_correlation <= correlation_threshold:
            selected_assets.append(asset)

    return selected_assets

In [5]:
selected_assets = select_diversified_assets(
    returns,
    correlation_threshold=0.70
)

print("Selected diversified assets:")
print(selected_assets)

Selected diversified assets:
['Stock_A', 'Stock_C']


5. Complete reusable function

For your portfolio-analysis project, I would structure it as a reusable function:

In [6]:
import pandas as pd


def spearman_portfolio_analysis(
    returns,
    correlation_threshold=0.70
):
    """
    Calculate Spearman rank correlation between
    portfolio/asset returns and make diversification
    decisions.
    """

    # -----------------------------------------
    # Step 1: Calculate Spearman correlation
    # -----------------------------------------

    spearman_matrix = returns.corr(
        method="spearman"
    )

    # -----------------------------------------
    # Step 2: Pairwise analysis
    # -----------------------------------------

    assets = spearman_matrix.columns

    pair_results = []

    for i in range(len(assets)):

        for j in range(i + 1, len(assets)):

            asset_1 = assets[i]
            asset_2 = assets[j]

            rho = spearman_matrix.loc[
                asset_1,
                asset_2
            ]

            # Make decision
            if rho > 0.70:

                decision = "REJECT / LIMITED DIVERSIFICATION"

            elif rho > 0.30:

                decision = "MODERATE DIVERSIFICATION"

            elif rho >= -0.30:

                decision = "GOOD DIVERSIFICATION"

            else:

                decision = "STRONG DIVERSIFICATION"

            pair_results.append({
                "Asset_1": asset_1,
                "Asset_2": asset_2,
                "Spearman_Rho": round(rho, 3),
                "Decision": decision
            })

    pair_analysis = pd.DataFrame(pair_results)

    # -----------------------------------------
    # Step 3: Select diversified assets
    # -----------------------------------------

    selected_assets = []

    for asset in assets:

        if not selected_assets:

            selected_assets.append(asset)
            continue

        correlations = spearman_matrix.loc[
            asset,
            selected_assets
        ]

        max_abs_correlation = correlations.abs().max()

        if max_abs_correlation <= correlation_threshold:

            selected_assets.append(asset)

    return (
        spearman_matrix,
        pair_analysis,
        selected_assets
    )

In [7]:
spearman_matrix, pair_analysis, selected_assets = \
    spearman_portfolio_analysis(
        returns,
        correlation_threshold=0.70
    )

print("========== SPEARMAN CORRELATION ==========")
print(spearman_matrix)

print("\n========== PAIRWISE DECISIONS ==========")
print(pair_analysis)

print("\n========== SELECTED ASSETS ==========")
print(selected_assets)

========== SPEARMAN CORRELATION ==========
          Stock_A   Stock_B   Stock_C   Stock_D   Stock_E
Stock_A  1.000000  1.000000  0.127273 -1.000000 -0.721212
Stock_B  1.000000  1.000000  0.127273 -1.000000 -0.721212
Stock_C  0.127273  0.127273  1.000000 -0.127273  0.224242
Stock_D -1.000000 -1.000000 -0.127273  1.000000  0.721212
Stock_E -0.721212 -0.721212  0.224242  0.721212  1.000000

========== PAIRWISE DECISIONS ==========
   Asset_1  Asset_2  Spearman_Rho                          Decision
0  Stock_A  Stock_B         1.000  REJECT / LIMITED DIVERSIFICATION
1  Stock_A  Stock_C         0.127              GOOD DIVERSIFICATION
2  Stock_A  Stock_D        -1.000            STRONG DIVERSIFICATION
3  Stock_A  Stock_E        -0.721            STRONG DIVERSIFICATION
4  Stock_B  Stock_C         0.127              GOOD DIVERSIFICATION
5  Stock_B  Stock_D        -1.000            STRONG DIVERSIFICATION
6  Stock_B  Stock_E        -0.721            STRONG DIVERSIFICATION
7  Stock_C  Stock_D    